# WM-811K Near-full 전역 형태 분석

고정 test의 Near-full 21개에서 결함 면적, 연결성, 중심/가장자리 분포, 경계 점유를 계산합니다. 모델·임계값은 변경하지 않으며 GPU는 필요하지 않습니다.

In [ ]:
!pip install -q pandas==2.2.3 matplotlib==3.10.0
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, sys
drive.mount('/content/drive')
print('이 분석은 CPU 런타임으로 충분합니다.')

아래 셀은 파일을 한 번에 하나씩 선택합니다. 프로젝트의 wafer_shape.py, analyze_wm811k_near_full_shape.py, split_assignments.csv를 차례대로 선택하세요.

In [ ]:
def upload_one(label, target, suffix):
    print(label)
    uploaded = files.upload()
    assert len(uploaded) == 1, '파일을 한 개만 선택하세요.'
    name, payload = next(iter(uploaded.items()))
    assert name.lower().endswith(suffix), f'{suffix} 파일을 선택하세요.'
    path = Path(target)
    path.write_bytes(payload)
    print('저장:', path)
    return path
SHAPE_MODULE = upload_one('1/3: wafer_shape.py를 선택하세요.', '/content/wafer_shape.py', '.py')
SCRIPT = upload_one('2/3: analyze_wm811k_near_full_shape.py를 선택하세요.', '/content/analyze_wm811k_near_full_shape.py', '.py')
ASSIGNMENTS = upload_one('3/3: split_assignments.csv를 선택하세요.', '/content/split_assignments.csv', '.csv')

In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
MAPS = DATA_DIR / 'wafer_maps_64.npy'
LABELS = DATA_DIR / 'labels.npy'
for path in (MAPS, LABELS, ASSIGNMENTS, SHAPE_MODULE, SCRIPT):
    assert path.is_file(), f'필수 파일이 없습니다: {path}'
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_near_full_shape/reference_run')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [sys.executable, str(SCRIPT), '--maps', str(MAPS), '--labels', str(LABELS), '--assignments', str(ASSIGNMENTS), '--output-dir', str(OUTPUT_DIR)]
subprocess.run(command, check=True)
print('전역 형태 분석 완료:', OUTPUT_DIR)

In [ ]:
ZIP_BASE = Path('/content/near_full_global_shape_results')
zip_path = Path(shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=OUTPUT_DIR))
print('다운로드 파일:', zip_path, f'{zip_path.stat().st_size / 1024**2:.2f} MB')
files.download(str(zip_path))